# Cloud IAM Risk Scoring

Rank synthetic cloud identities by privilege, exposure, credential hygiene, and suspicious activity.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Train an explainable risk model and measure how many risky identities appear within the top review decile.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def sigmoid(values):
    clipped = np.clip(values, -30, 30)
    return 1.0 / (1.0 + np.exp(-clipped))

def split_indices(size, test_fraction=0.25):
    shuffled = rng.permutation(size)
    split_at = int(size * (1 - test_fraction))
    return shuffled[:split_at], shuffled[split_at:]

def standardize(train_values, test_values):
    mean = train_values.mean(axis=0)
    std = train_values.std(axis=0)
    std = np.where(std < 1e-9, 1.0, std)
    return (train_values - mean) / std, (test_values - mean) / std, mean, std

def fit_logistic(features, labels, steps=1400, learning_rate=0.08, l2=0.01):
    design = np.column_stack([np.ones(len(features)), features])
    weights = np.zeros(design.shape[1])
    for _ in range(steps):
        probabilities = sigmoid(design @ weights)
        gradient = design.T @ (probabilities - labels) / len(labels)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient
    return weights

def predict_probability(features, weights):
    design = np.column_stack([np.ones(len(features)), features])
    return sigmoid(design @ weights)

def classification_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    accuracy = (tp + tn) / max(len(labels), 1)
    return pd.Series({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    })


## Steps

### 1. Generate synthetic IAM posture data


In [2]:
principal_count = 1900
admin_privilege = rng.binomial(1, 0.09, principal_count)
wildcard_actions = rng.poisson(0.6, principal_count)
wildcard_resources = rng.poisson(0.8, principal_count)
external_trust = rng.binomial(1, 0.11, principal_count)
mfa_disabled = rng.binomial(1, 0.16, principal_count)
access_key_age_days = rng.gamma(2.0, 55.0, principal_count).clip(0, 500)
unused_days = rng.gamma(1.7, 35.0, principal_count).clip(0, 365)
anomalous_api_calls = rng.poisson(0.5, principal_count)

risky_probability = sigmoid(
    -5.7
    + 1.55 * admin_privilege
    + 0.42 * wildcard_actions
    + 0.32 * wildcard_resources
    + 1.05 * external_trust
    + 0.90 * mfa_disabled
    + 0.004 * access_key_age_days
    + 0.003 * unused_days
    + 0.55 * anomalous_api_calls
)
confirmed_risky = rng.binomial(1, risky_probability)

iam_principals = pd.DataFrame({
    "admin_privilege": admin_privilege,
    "wildcard_actions": wildcard_actions,
    "wildcard_resources": wildcard_resources,
    "external_trust": external_trust,
    "mfa_disabled": mfa_disabled,
    "access_key_age_days": access_key_age_days,
    "unused_days": unused_days,
    "anomalous_api_calls": anomalous_api_calls,
    "confirmed_risky": confirmed_risky,
})

print("Principal count:", len(iam_principals))
print("Confirmed-risk rate:", round(iam_principals["confirmed_risky"].mean(), 3))
print(iam_principals.head(6).round(2).to_string(index=False))


Principal count: 1900
Confirmed-risk rate: 0.026
 admin_privilege  wildcard_actions  wildcard_resources  external_trust  mfa_disabled  access_key_age_days  unused_days  anomalous_api_calls  confirmed_risky
               0                 1                   1               0             0               103.22        64.70                    1                0
               0                 1                   0               0             0                44.92        46.36                    0                0
               0                 1                   2               1             0               176.67        13.47                    0                0
               0                 0                   0               0             0                33.33        17.30                    0                0
               0                 0                   1               0             0               112.35        66.95                    0                0
         

### 2. Model, rank, and explain identity risk


In [3]:
feature_names = [column for column in iam_principals.columns if column != "confirmed_risky"]
train_rows, test_rows = split_indices(len(iam_principals))
train_values = iam_principals.loc[train_rows, feature_names].to_numpy(float)
test_values = iam_principals.loc[test_rows, feature_names].to_numpy(float)
train_labels = iam_principals.loc[train_rows, "confirmed_risky"].to_numpy(int)
test_labels = iam_principals.loc[test_rows, "confirmed_risky"].to_numpy(int)

train_scaled, test_scaled, _, _ = standardize(train_values, test_values)
iam_weights = fit_logistic(train_scaled, train_labels)
iam_probability = predict_probability(test_scaled, iam_weights)
iam_prediction = (iam_probability >= 0.5).astype(int)
iam_metrics = classification_metrics(test_labels, iam_prediction)

ranked_identities = iam_principals.loc[test_rows].copy()
ranked_identities["risk_probability"] = iam_probability
ranked_identities = ranked_identities.sort_values("risk_probability", ascending=False)
review_count = max(1, int(len(ranked_identities) * 0.10))
top_decile_recall = (
    ranked_identities.head(review_count)["confirmed_risky"].sum()
    / max(ranked_identities["confirmed_risky"].sum(), 1)
)
iam_importance = pd.DataFrame({
    "feature": feature_names,
    "standardized_weight": iam_weights[1:],
}).sort_values("standardized_weight", ascending=False)

print("Test metrics:")
print(iam_metrics.round(3).to_string())
print("\nRisk captured in top review decile:", round(top_decile_recall, 3))
print("\nFeature weights:")
print(iam_importance.round(3).to_string(index=False))
print("\nHighest-risk identities:")
print(ranked_identities.head(8).round(3).to_string(index=False))


Test metrics:
accuracy       0.971
precision      0.000
recall         0.000
f1             0.000
tp             0.000
fp             0.000
tn           461.000
fn            14.000

Risk captured in top review decile: 0.571

Feature weights:
            feature  standardized_weight
   wildcard_actions                0.355
access_key_age_days                0.336
 wildcard_resources                0.309
    admin_privilege                0.255
anomalous_api_calls                0.211
     external_trust                0.130
       mfa_disabled                0.126
        unused_days                0.060

Highest-risk identities:
 admin_privilege  wildcard_actions  wildcard_resources  external_trust  mfa_disabled  access_key_age_days  unused_days  anomalous_api_calls  confirmed_risky  risk_probability
               1                 3                   1               1             0              140.410      304.837                    0                0             0.229
            

## Checks


In [4]:
assert 0.02 < iam_principals["confirmed_risky"].mean() < 0.50
assert top_decile_recall >= 0.30
assert ranked_identities["risk_probability"].is_monotonic_decreasing
assert ranked_identities["risk_probability"].between(0, 1).all()
print("Checks passed: plausible class balance, useful review concentration, and bounded sorted scores.")


Checks passed: plausible class balance, useful review concentration, and bounded sorted scores.


## Next Steps

        - Map features to AWS, Azure, or GCP identity and policy schemas.
- Add toxic permission combinations and resource sensitivity.
- Separate remediation priority from model probability and require owner review before changes.
